# 🤖 AI-Publisher Colab Docker Kurulumu

Bu notebook, AI-Publisher platformunun tüm model konteynerlarını Google Colab'da başlatır.

### 📋 Kurulum Adımları
1. **Runtime kontrolü** - GPU durumunu doğrula
2. **Bağımlılıklar** - Drive mount ve pip paketleri
3. **Kaniko + Registry + pigz** - Daemonless build araçları (CPU)
4. **Repo güncelleme** - GitHub'dan en son kodları çek
5. **İmaj yükleme** - Drive'dan yükle veya sıfırdan build et
6. **Shared dizin hazırlığı** - lazy-loading için gerekli klasörler
7. **Ngrok tüneli** - Dış dünyaya aç
8. **Sunucu başlat** - colab_server.py + canlı log

> ⚠️ **İki Aşamalı Çalışma**:
> **Aşama 1 (BUILD — CPU, bu notebook):** İmajları Kaniko ile build et, Drive'a yedekle.
> **Aşama 2 (RUN — GPU, ayrı oturum):** İmajları Drive'dan yükle, Docker daemon + colab_server.py ile çalıştır.
>

---
Her hücreyi sırasıyla çalıştırın. İlk çalıştırma sonrası oturum yeniden başlatılabilir.

## Hücre 1: Runtime Kontrolü

GPU durumunu kontrol et. Bu notebook T4 GPU ile çalışır.
CPU-only modeller (Whisper, KokoroTTS) GPU olmadan da çalışır.

> **İki Aşamalı Çalışma:**
> **Aşama 1 (BUILD — CPU, bu notebook):** İmajları Kaniko ile build et, Drive'a yedekle.
> **Aşama 2 (RUN — GPU, ayrı oturum):** İmajları Drive'dan yükle, Docker daemon + colab_server.py ile çalıştır.

**Notebook Akışı:**
1. **Runtime Kontrolü** - GPU/CPU durumu
2. **Bağımlılıklar ve Drive Mount** - Google Drive bağla, gerekli paketler
3. **Kaniko + Registry + pigz** - Daemonless build araçları (CPU)
4. **Repo Güncelleme** - GitHub'dan son kodlar
5. **Docker İmaj Build** - 16 imajı Kaniko ile build et, Drive'a kaydet
6. **Shared Dizin** - Ortak dosya yapısı (lazy-loading containerlar)
7. **Ngrok Tüneli** - Colab'ı dış dünyaya aç
8. **Sunucu Başlat** - Flask gateway + ContainerManager çalıştır
9. **Canlı Log** - Sunucu loglarını izle


In [ ]:
#@title Runtime Kontrolü { display-mode: "form" }
#@markdown GPU durumunu kontrol et
import subprocess
import os

def check_runtime():
    """GPU ve sistem durumunu kontrol et."""
    print("=" * 60)
    print("🔍 AI-Publisher Runtime Kontrolü")
    print("=" * 60)
    
    # GPU kontrolü
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(", ")
            print(f"✅ GPU: {gpu_info[0]}")
            print(f"   Toplam VRAM: {gpu_info[1]} MB")
            print(f"   Boş VRAM: {gpu_info[2]} MB")
        else:
            print("❌ GPU bulunamadı! Runtime > Change runtime type > GPU seçin.")
            return False
    except FileNotFoundError:
        print("❌ nvidia-smi bulunamadı. GPU runtime aktif değil.")
        return False
    
    # Disk alanı
    disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
    if disk.returncode == 0:
        lines = disk.stdout.strip().split("\n")
        if len(lines) > 1:
            parts = lines[1].split()
            print(f"💾 Disk: {parts[2]} kullanıldı / {parts[1]} toplam")
    
    # RAM
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                total_kb = int(line.split()[1])
                print(f"🧠 RAM: {total_kb // 1024} MB")
                break
    
    print("\n✅ Runtime hazır. Sonraki hücreye geçin.")
    return True

check_runtime()

## 📌 Hücre 2: Bağımlılıklar ve Drive Mount

Google Drive'ı bağla ve gerekli Python paketlerini kur.
Docker imajları Drive'da önbelleklenir, her seferinde yeniden build edilmez.

In [ ]:
#@title Bağımlılıklar ve Drive Mount { display-mode: "form" }
import subprocess
import sys
import os
import time

print("📦 Bağımlılıklar kuruluyor...")

# Google Drive mount
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
        print("✅ Google Drive bağlandı.")
    else:
        print("✅ Google Drive zaten bağlı.")
except Exception as e:
    print(f"⚠️ Drive mount hatası: {e}")

# Python bağımlılıkları
deps = [
    "flask",
    "requests",
    "pyngrok",
    "opencv-python-headless",
    "numpy<2.0.0",
    "yt-dlp",
]

for pkg in deps:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"  ⚠️ {pkg} kurulamadi, tek deneniyor...")
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f"  ❌ {pkg} KURULAMADI!")
            print(f"     {r.stderr[-200:]}")
        else:
            print(f"  ✅ {pkg} kuruldu")
    else:
        print(f"  ✅ {pkg} kuruldu")

print("✅ Python bağımlılıkları kuruldu.")

# Drive çıktı dizinini oluştur
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Çıktı dizini hazır: {OUTPUT_DIR}")

print("\n✅ Hücre 2 tamamlandı.")

## 📌 Hücre 3: Kaniko + Registry + pigz Kurulumu

Daemonless Docker imaj build için Kaniko + yerel registry kurulur.
GPU gerekmez — CPU runtime'da çalışır.
Bu işlem sadece ilk çalıştırmada yapılır.

In [ ]:
#@title Kaniko + Registry + pigz Kurulumu { display-mode: "form" }
import subprocess
import shutil
import sys
import os
import time

def run(cmd, label="", retries=3, timeout=300):
    for attempt in range(1, retries + 1):
        try:
            result = subprocess.run(
                cmd, shell=True, check=True,
                capture_output=True, text=True, timeout=timeout
            )
            return True
        except Exception as e:
            if attempt == retries:
                print(f"  X {label} basarisiz ({retries} deneme): {e}")
                return False
            time.sleep(3)
    return False

print("Kaniko kuruluyor...")
if shutil.which("kaniko"):
    print("Kaniko zaten kurulu.")
elif os.path.exists("/usr/local/bin/kaniko"):
    print("Kaniko zaten kurulu.")
else:
    # crane ile Kaniko imajindan binary cikar
    print("  crane indiriliyor...")
    run("curl -Lo /tmp/crane.tar.gz https://github.com/google/go-containerregistry/releases/download/v0.20.2/go-containerregistry_Linux_x86_64.tar.gz", label="crane indirme", timeout=120)
    run("tar -xzf /tmp/crane.tar.gz -C /usr/local/bin/ crane", label="crane cikarma")
    subprocess.run(["rm", "-f", "/tmp/crane.tar.gz"])
    print("  Kaniko binary imajdan cikariliyor...")
    run("/usr/local/bin/crane export gcr.io/kaniko-project/executor:v1.23.2 - | tar -xOf - kaniko/executor > /usr/local/bin/kaniko", label="Kaniko cikarma", timeout=120)
    subprocess.run(["chmod", "+x", "/usr/local/bin/kaniko"])
    result = subprocess.run(["/usr/local/bin/kaniko", "version"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print(f"  Kaniko kuruldu: {result.stdout.strip()}")
    else:
        print("  Uyari: Kaniko binary cikarildi ancak dogrulama basarisiz.")

print("\nYerel Registry kuruluyor...")
if shutil.which("registry"):
    print("Registry binary zaten kurulu.")
else:
    run("wget -q https://github.com/distribution/distribution/releases/download/v2.8.2/registry_2.8.2_linux_amd64.tar.gz", label="Registry indirme")
    run("tar -xzf registry_2.8.2_linux_amd64.tar.gz registry", label="Registry cikarma")
    subprocess.run(["mv", "registry", "/usr/local/bin/registry"], capture_output=True)
    subprocess.run(["rm", "-f", "registry_2.8.2_linux_amd64.tar.gz"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/registry"])
    print("Registry binary kuruldu.")

# Registry'yi baslat
subprocess.run("pkill -f 'registry serve'", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
os.makedirs("/etc/docker/registry", exist_ok=True)
with open("/etc/docker/registry/config.yml", "w") as cfg:
    cfg.write("version: 0.1\nlog:\n  fields:\n    service: registry\nstorage:\n  cache:\n    blobdescriptor: inmemory\n  filesystem:\n    rootdirectory: /var/lib/registry\nhttp:\n  addr: :5000\n  headers:\n    X-Content-Type-Options: [nosniff]\n")
subprocess.Popen(
    ["/usr/local/bin/registry", "serve", "/etc/docker/registry/config.yml"],
    stdout=open("registry.log", "w"), stderr=subprocess.STDOUT
)
print("Registry hazir bekleniyor...")
import urllib.request
registry_ok = False
for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:5000/v2/")
        registry_ok = True
        break
    except:
        time.sleep(1)
if not registry_ok:
    print("Registry baslatilamadi!")
    if os.path.exists("registry.log"):
        with open("registry.log") as f:
            print(f.read())
else:
    print("Registry localhost:5000 hazir.")

print("\npigz kuruluyor...")
run("apt-get install -y -q pigz", label="pigz")

print("\nHuc 3 tamamlandi.")


## 📌 Hücre 4: Repo Güncelleme

GitHub'dan en son kodları çek veya mevcut repo'yu güncelle.

In [ ]:
#@title Repo Guncelle { display-mode: "form" }
#@markdown GitHub token'iniz Colab Secrets'ta `GITHUB_TOKEN` olarak tanimli olmali.
#@markdown (Sag panel > Secrets > `GITHUB_TOKEN` ekleyin)
#@markdown Ya da asagidaki alana token girebilirsiniz:
MANUAL_TOKEN = "" #@param {type:"string"}

import subprocess
import os
import shutil
import sys

REPO_URL = "https://github.com/Arda-Avci/AI-Publisher.git"
REPO_DIR = "/content/AI-Publisher"

GH_TOKEN = MANUAL_TOKEN.strip()
if not GH_TOKEN:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get("GITHUB_TOKEN")
        print("[OK] Token Colab Secrets'tan alindi.")
    except Exception:
        print("[UYARI] Colab Secrets'ta GITHUB_TOKEN bulunamadi. Manuel giris deneniyor...")

# Onceki kirli klonu temizle
if os.path.exists(REPO_DIR):
    print("[TEMIZLIK] Eski repo dizini temizleniyor...")
    shutil.rmtree(REPO_DIR, ignore_errors=True)

# Token ile klonla
if GH_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")
    print("[BILGI] GitHub token ile klonlaniyor...")
else:
    clone_url = REPO_URL
    print("[UYARI] Token yok, public repo deneniyor...")

print(f"Repo klonlaniyor...")
result = subprocess.run(
    ["git", "clone", clone_url, REPO_DIR],
    capture_output=True, text=True, timeout=120
)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    print("\nHATA: Repo klonlanamadi!")
    print("Token gecerli mi kontrol edin veya Colab Secrets > GITHUB_TOKEN ekleyin.")
    print("\nAlternatif - manuel:")
    print(f"!git clone https://TOKEN@github.com/Arda-Avci/AI-Publisher.git {REPO_DIR}")
    sys.exit(1)

print("Repo basariyla klonlandi.")

# Git LFS dosyalarini cek
print("\nLFS dosyalari cekiliyor...")
subprocess.run(["git", "-C", REPO_DIR, "lfs", "install"], capture_output=True, text=True)
subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull"], capture_output=True, text=True)
print("LFS dosyalari cekildi.")

# colab_docker dizinini kontrol et
docker_dir = os.path.join(REPO_DIR, "colab_docker")
if os.path.exists(docker_dir):
    models = [d for d in os.listdir(docker_dir) if os.path.isdir(os.path.join(docker_dir, d))]
    print(f"\nBulunan konteyner dizinleri ({len(models)}):")
    for m in sorted(models):
        print(f"   {m}")
else:
    print("colab_docker dizini bulunamadi!")
    print(f"Repo icerigi: {os.listdir(REPO_DIR)}")

print("\nHucre 4 tamamlandi.")


## 📌 Hücre 5: Docker İmaj Build (Kaniko + Drive Yedek)

Docker imajlarını Kaniko ile sıralı build edip Drive'a kaydeder.
GPU gerekmez — CPU runtime'da çalışır.
Drive'da hazır imaj varsa atlanır, sadece eksikler build edilir.

> 🕐 İlk build ~30-45 dk sürebilir.
> Sonraki çalıştırmalarda sadece eksik imajlar build edilir.

In [ ]:
#@title Docker İmaj Build (Kaniko + Registry) { display-mode: "form" }
import subprocess
import os
import sys
import time
import shutil
import urllib.request

# Registry kontrolu - calismiyorsa baslat
registry_ok = False
try:
    urllib.request.urlopen("http://localhost:5000/v2/", timeout=3)
    registry_ok = True
except:
    pass

if not registry_ok:
    if shutil.which("registry"):
        print("🔧 Registry baslatiliyor...")
        subprocess.run("pkill -f 'registry serve' || true", shell=True, timeout=5)
        subprocess.Popen(
            ["registry", "serve", "/etc/docker/registry/config.yml"],
            stdout=open("registry.log", "w"), stderr=subprocess.STDOUT
        )
        for _ in range(15):
            try:
                urllib.request.urlopen("http://localhost:5000/v2/")
                print("✅ Registry localhost:5000 hazir!")
                registry_ok = True
                break
            except:
                time.sleep(1)
    if not registry_ok:
        print("❌ Registry baslatilamadi! Önce Hücre 3'u calistirin.")
        sys.exit(1)

REPO_DIR = "/content/AI-Publisher"
BUILD_DIR = os.path.join(REPO_DIR, "colab_docker")

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
os.makedirs(DRIVE_IMAGES_DIR, exist_ok=True)

ALL_MODELS = [
    "base",  # Dockerfile.base -> ai-publisher-base:latest
    "cogvideox", "wan", "ltx", "hunyuan",
    "wan25", "svd", "animatediff",
    "xtts", "f5tts", "kokorotts",
    "audioldm2",
    "wav2lip", "musetalk",
    "whisper",
    "stablediffusion",
    "lora-trainer",
]

# Drive'da mevcut imajlari listele (sadece bilgi)
print("🔍 Drive'da mevcut imajlar taranıyor...")
existing = [m for m in ALL_MODELS if os.path.exists(os.path.join(DRIVE_IMAGES_DIR, f"{m}.tar.gz"))]
missing = [m for m in ALL_MODELS if not os.path.exists(os.path.join(DRIVE_IMAGES_DIR, f"{m}.tar.gz"))]
print(f"   Drive'da: {len(existing)}/{len(ALL_MODELS)} imaj var")
print(f"   Build gerekli: {len(missing)} imaj (atlanacak: {len(existing)} imaj Drive'da hazir)")
print("   docker load atlandi — build_all.sh zaten sadece eksikleri build eder.")

# Sadece eksik olanlari build et
if missing:
    print(f"\n🔨 {len(missing)} imaj Kaniko ile build edilecek...")
    
    # build_all.sh calistir
    build_script = os.path.join(BUILD_DIR, "build_all.sh")
    if not os.path.exists(build_script):
        print(f"❌ {build_script} bulunamadi!")
    else:
        subprocess.run(["chmod", "+x", build_script])
        print(f"\n🚀 build_all.sh başlatılıyor...")
        start = time.time()
        
        process = subprocess.Popen(
            ["bash", build_script],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=BUILD_DIR
        )
        for line in process.stdout:
            print(line.rstrip())
        process.wait()
        
        elapsed = time.time() - start
        if process.returncode == 0:
            print(f"\n✅ Tüm imajlar build edildi ve Drive'a kaydedildi ({elapsed:.0f}s)")
        else:
            print(f"\n❌ Build hatası (exit code: {process.returncode})")
else:
    print("\n✅ Tüm imajlar Drive'da mevcut. Build gerekmiyor.")
print("\n✅ Hücre 5 tamamlandı.")

## 📌 Hücre 6: Shared Dizin ve Container Hazırlığı

Konteynerlar burada başlatılmaz. **Lazy-Loading** mimarisi:
- `colab_server.py` içindeki **ContainerManager**, ihtiyaç duyulan konteyneri `docker run` ile otomatik başlatır
- T4 GPU (15GB VRAM) tüm GPU konteynerlarını aynı anda kaldıramaz → ContainerManager VRAM tasarrufu için:
  1. Yeni bir GPU konteyneri gerektiğinde, eski GPU konteynerini `docker stop` ile durdurur
  2. CPU-only konteynerlar (whisper, kokorotts) her zaman ayakta kalabilir
  3. 2 dakika boş kalan konteyner otomatik durdurulur
  4. 5 dakika boş kalan VM otomatik kapatılır

Bu hücre sadece shared dizinini ve çıktı klasörlerini hazırlar.

In [ ]:
#@title Shared Dizin ve Çıktı Klasörlerini Hazırla { display-mode: "form" }
import os

BUILD_DIR = "/content/AI-Publisher/colab_docker"
SHARED_DIR = os.path.join(BUILD_DIR, "shared")
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/outputs"

os.makedirs(SHARED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Shared dizin: {SHARED_DIR}")
print(f"✅ Çıktı dizini: {OUTPUT_DIR}")
print("\n📌 Konteynerlar başlatılmadı — colab_server.py ContainerManager")
print("   ihtiyaç duydukça `docker run` ile lazy-loading yapacak.")
print("   Hücre 8'de sunucu başlayınca otomatik devreye girer.")

print("\n✅ Hücre 6 tamamlandı.")

## 📌 Hücre 7: Ngrok Tüneli

Colab'ı dış dünyaya açar. Ngrok token'ınız gerekir.
> 🔑 [ngrok.com](https://ngrok.com) adresinden ücretsiz token alabilirsiniz.

In [ ]:
#@title Ngrok Tüneli Kur { display-mode: "form" }
#@markdown Ngrok Auth Token'ınızı girin
NGROK_TOKEN = "" #@param {type:"string"}

import os
import subprocess
import time

if not NGROK_TOKEN:
    # Colab userdata'dan dene
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get("NGROK_TOKEN")
    except:
        pass

if not NGROK_TOKEN:
    print("❌ NGROK_TOKEN gerekli! Hücreye yapıştırın veya Colab Secrets'a ekleyin.")
    print("   https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    # Eski ngrok süreçlerini temizle
    subprocess.run("pkill -9 ngrok", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    
    # Ngrok config ve başlat
    os.makedirs(os.path.expanduser("~/.ngrok2"), exist_ok=True)
    with open(os.path.expanduser("~/.ngrok2/ngrok.yml"), "w") as f:
        f.write(f"authtoken: {NGROK_TOKEN}\n")
    
    # Ngrok tünelini başlat (port 5000 = colab_server.py)
    subprocess.Popen(
        ["ngrok", "http", "5000", "--log=stdout"],
        stdout=open("/content/ngrok.log", "w"),
        stderr=subprocess.STDOUT
    )
    
    # URL bekle
    print("⏳ Ngrok URL bekleniyor...")
    ngrok_url = None
    for _ in range(30):
        time.sleep(2)
        try:
            import requests
            resp = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=5)
            data = resp.json()
            if data.get("tunnels"):
                ngrok_url = data["tunnels"][0]["public_url"]
                break
        except:
            pass
    
    if ngrok_url:
        print(f"\n🔗 NGROK URL (Node.js projenize yapıştırın):")
        print(f"\n   {ngrok_url}\n")
        # URL'yi dosyaya kaydet
        with open("/content/ngrok_url.txt", "w") as f:
            f.write(ngrok_url)
    else:
        print("❌ Ngrok URL alınamadı. Token'ı kontrol edin.")
        print("   Logs: /content/ngrok.log")

print("\n✅ Hücre 7 tamamlandı.")

## 📌 Hücre 8: Sunucu Başlat ve Canlı Log

colab_server.py'yi başlatır (Flask gateway + ContainerManager).
Bu sunucu:
- Node.js istemciden gelen istekleri alır
- ContainerManager ile ihtiyaç duyulan Docker konteynerini lazy başlatır
- VRAM yönetimi yapar (eski GPU konteynerini durdurur)
- FFmpeg/OpenCV işlemlerini host üzerinde yürütür

In [ ]:
#@title Sunucu Başlat { display-mode: "form" }
import subprocess
import os
import sys
import time

REPO_DIR = "/content/AI-Publisher"
SERVER_SCRIPT = os.path.join(REPO_DIR, "colab_server.py")
LOG_FILE = "/content/colab_server.log"

# colab_server.py mevcut mu?
if not os.path.exists(SERVER_SCRIPT):
    print(f"❌ colab_server.py bulunamadı: {SERVER_SCRIPT}")
else:
    # Eski süreci durdur
    subprocess.run("pkill -f colab_server.py", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    
    # Ngrok URL'sini al
    ngrok_url = ""
    if os.path.exists("/content/ngrok_url.txt"):
        with open("/content/ngrok_url.txt") as f:
            ngrok_url = f.read().strip()
    
    # Sunucu env
    env = os.environ.copy()
    env["NGROK_URL"] = ngrok_url
    
    # colab_server.py'yi başlat
    print("🚀 colab_server.py başlatılıyor...")
    with open(LOG_FILE, "w") as log:
        subprocess.Popen(
            [sys.executable, "-u", SERVER_SCRIPT],
            stdout=log,
            stderr=subprocess.STDOUT,
            env=env,
        )
    
    # Başlatılmasını bekle
    time.sleep(5)
    if os.path.exists(LOG_FILE) and os.path.getsize(LOG_FILE) > 0:
        print("✅ Sunucu başlatıldı.")
        print("\n📋 Son log satırları:")
        with open(LOG_FILE) as f:
            lines = f.readlines()
            for line in lines[-20:]:
                print(f"   {line.rstrip()}")
    else:
        print("⚠️ Sunucu henüz log üretmedi. Biraz bekleyin.")

print("\n✅ Hücre 8 tamamlandı.")
print("\n" + "=" * 60)
print("🎉 KURULUM TAMAMLANDI!")
print("=" * 60)

## 📌 Hücre 9: Canlı Log İzleme (İsteğe Bağlı)

Sunucu loglarını canlı olarak izlemek için bu hücreyi çalıştırın.
Clear output ile durdurabilirsiniz.

In [ ]:
#@title Canlı Log İzleme { display-mode: "form" }
#@markdown Clear output ile durdurun
import time
import os

LOG_FILE = "/content/colab_server.log"

if not os.path.exists(LOG_FILE):
    print("❌ Log dosyası bulunamadı. Önce Hücre 8'i çalıştırın.")
else:
    print("📋 Canlı loglar (Clear output ile durdurun):")
    print("-" * 60)
    with open(LOG_FILE) as f:
        # Dosya sonuna git
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line.rstrip())
            else:
                time.sleep(1)